## Project Explanation: Side-Channel Attack on RSA using Deep Learning

This project demonstrates a simplified side-channel attack targeting the RSA cryptosystem's modular exponentiation process. A side-channel attack exploits information leaked from the physical implementation of a cryptosystem, such as power consumption, electromagnetic emissions, or timing variations. In this simulation, we focus on power consumption.

RSA's security relies on the difficulty of factoring large numbers. However, the calculation of `base^exponent mod n` (where `exponent` is often the private key) can reveal information about the private key's bits if not implemented carefully.

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


### The 'Square-and-Multiply' Algorithm (`power_mod` function)

The `power_mod` function simulates the Square-and-Multiply algorithm, a common method for modular exponentiation. It works by iterating through the bits of the exponent:

1.  **Square Step**: For every bit, a 'square' operation `res = (res * res) % n` is performed. This operation has a baseline power consumption.
2.  **Multiply Step**: If the current bit of the exponent is '1', an additional 'multiply' operation `res = (res * base) % n` is performed. This operation consumes more power than the square step.

This difference in power consumption between a '0' bit (only square) and a '1' bit (square + multiply) creates a "leakage" that can be observed. Our `trace` list captures this simulated power consumption for each step, with added noise to make it more realistic.

In [6]:
def power_mod(base, exponent, n):
    """
    Square-and-Multiply implementation.
    This is where the information 'leaks' physically.
    """
    res = 1
    base = base % n
    trace = [] # To store the energy 'leakage'

    # We iterate through the bits of the exponent (private key)
    for bit in bin(exponent)[2:]:
        # 1. 'SQUARE' step (Always present)
        res = (res * res) % n
        trace.append(1.0 + np.random.normal(0, 0.05)) # Base consumption

        # 2. 'MULTIPLY' step (Only if the bit is 1)
        if bit == '1':
            res = (res * base) % n
            trace.append(1.5 + np.random.normal(0, 0.05)) # Overconsumption

    return res, trace

### Data Generation (`generate_dataset` function)

The `generate_dataset` function creates synthetic power traces for training our AI. It simulates two distinct power patterns:

*   **For a '0' bit**: A short signal representing only the 'square' operation followed by 'rest' periods.
*   **For a '1' bit**: A longer signal representing both the 'square' and 'multiply' operations, also followed by 'rest' periods.

Crucially, random noise is added to these signals to mimic real-world measurement challenges, making the task harder for the AI, similar to how actual side-channel traces are noisy.

In [7]:
def generate_dataset(num_samples=1000):
    X = [] # Electrical traces
    y = [] # Actual bits

    for _ in range(num_samples):
        # Simulate either a 0 bit (one operation) or a 1 bit (two operations)
        bit = np.random.randint(0, 2)
        if bit == 0:
            # Short signal for a '0'
            sig = [1.0] + [0.1]*5 # Operation + rest
        else:
            # Long signal for a '1'
            sig = [1.0, 1.5] + [0.1]*4 # Two operations + rest

        # Make the signal complex for the AI (noise + shift)
        sig = np.array(sig) + np.random.normal(0, 0.08, len(sig))
        X.append(sig)
        y.append(bit)

    return np.array(X).reshape(-1, 6, 1), np.array(y)

### AI Model (`build_model` function)

We use a simple Convolutional Neural Network (CNN) to act as our "spy" AI. The model is designed to:

1.  **`Conv1D` Layer**: Detect patterns in the 1D power trace data.
2.  **`Flatten` Layer**: Prepare the data for the dense layers.
3.  **`Dense` Layers**: Classify the power traces as corresponding to either a '0' or a '1' bit. The final `sigmoid` activation outputs a probability for the bit being '1'.

The model is trained to distinguish between the power patterns of '0's and '1's, effectively learning to "read" the secret exponent bit by bit from the leaked power traces.

In [12]:
from tensorflow.keras import layers, models, Input

def build_model():
    model = models.Sequential([
        Input(shape=(6, 1)), # Recommended way to define input shape
        layers.Conv1D(8, 3, activation='relu'),
        layers.Flatten(),
        layers.Dense(8, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

### Attack Simulation

The overall process of the attack is:

1.  **Data Generation**: We generate a training and testing dataset of simulated power traces and their corresponding exponent bits.
2.  **AI Training**: The CNN model (`ai_spy`) is trained on these datasets to learn the correlation between power traces and exponent bits.
3.  **Attack Testing**: We evaluate the AI's accuracy on unseen data.
4.  **Practical Application**: We then simulate a real-world scenario where the `power_mod` function is executed with a known exponent (e.g., `0b101`). We extract the power traces (e.g., for the first and second bits) and feed them to our trained `ai_spy` model to predict the individual bits of the exponent. This demonstrates how an attacker could recover parts of a private key by analyzing physical leakages.

In [14]:
print("--- Side-Channel Attack Initialization ---")

# 1. Data Generation
X_train, y_train = generate_dataset(3000)
X_test, y_test = generate_dataset(500)

# 2. AI Training
print("\n[INFO] AI is learning to recognize RSA's power patterns...")
ai_spy = build_model()
ai_spy.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

# 3. Attack Testing
loss, accuracy = ai_spy.evaluate(X_test, y_test, verbose=0)
print(f"[RESULT] AI accuracy in guessing bits: {accuracy*100:.2f}%")

# 4. Practical Application
print("\n--- Simulation of an interception ---")
_, actual_leakage = power_mod(123, 0b101, 999) # Test on a '1' bit
print(f"Traces generated by RSA (noisy): {np.round(actual_leakage[:2], 2)}")

# Prediction for the first bit of the exponent (which is '1')
# The first two elements of actual_leakage represent the consumption for the Square and Multiply of the first bit.
# We need to add 4 "rests" for the trace to have a length of 6, like the training data.
sample_for_prediction_bit1 = actual_leakage[0:2] + [0.1] * 4
pred_bit1 = ai_spy.predict(np.array(sample_for_prediction_bit1).reshape(1, 6, 1), verbose=0)
print(f"The AI predicts the FIRST bit of the exponent is a: {'1' if pred_bit1 > 0.5 else '0'}")

# Example to predict the second bit of the exponent (which is '0')
# The third element of actual_leakage represents the consumption for the Square operation of the second bit.
# We need to add 5 "rests" to match the input shape (length 6).
sample_for_prediction_bit0 = actual_leakage[2:3] + [0.1] * 5
pred_bit0 = ai_spy.predict(np.array(sample_for_prediction_bit0).reshape(1, 6, 1), verbose=0)
print(f"The AI predicts the SECOND bit of the exponent is a: {'1' if pred_bit0 > 0.5 else '0'}")

--- Side-Channel Attack Initialization ---

[INFO] AI is learning to recognize RSA's power patterns...


[RESULT] AI accuracy in guessing bits: 100.00%

--- Simulation of an interception ---
Traces generated by RSA (noisy): [0.99 1.42]
The AI predicts the FIRST bit of the exponent is a: 1
The AI predicts the SECOND bit of the exponent is a: 0
